# Pilot v1.1 — CSV verification notebook

Load and inspect Pilot Release **v1.1** CSVs in a structured order.

For each file: path check → shape → column headers → first 5 rows.

**Suggested reading order:**
1. Identity & snapshot backbone
2. Page-level extraction + classification
3. Branding / NLP corpus (legal/technical excluded)
4. Impressum / governance layer
5. Observation-level text eligibility
6. Quality summary (renamed metrics)
7. Analysis pools & validation samples

## 0. Setup

In [1]:
from pathlib import Path

import pandas as pd

# Prefer the frozen release bundle; fall back to working output if needed.
PROJECT_ROOT = Path("..").resolve()
RELEASE_DIR = PROJECT_ROOT / "data" / "releases" / "pilot_v1_1" / "data"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

DATA_DIR = RELEASE_DIR if RELEASE_DIR.exists() else OUTPUT_DIR
print("Loading from:", DATA_DIR)
assert DATA_DIR.exists(), f"Missing data directory: {DATA_DIR}"

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

Loading from: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data


In [2]:
def load_csv(name: str) -> pd.DataFrame:
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def inspect(name: str, df: pd.DataFrame, highlight: list[str] | None = None) -> None:
    """Show headers + first 5 rows; optionally flag key columns."""
    print("=" * 72)
    print(f"FILE: {name}")
    print(f"path: {DATA_DIR / name}")
    print(f"shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print("\nColumns:")
    for i, col in enumerate(df.columns, 1):
        mark = "  ← key" if highlight and col in highlight else ""
        print(f"  {i:2d}. {col}{mark}")
    if highlight:
        missing = [c for c in highlight if c not in df.columns]
        if missing:
            print("\n⚠ Missing expected columns:", missing)
        else:
            print("\n✓ All highlighted columns present")
    print("\nFirst 5 rows:")
    display(df.head(5))

---
## 1. Identity & snapshot backbone

Who the firms are, and which firm × timepoint snapshots were selected.

| File | Role |
|------|------|
| `firms.csv` | 5 pilot firms |
| `snapshots.csv` | 25 firm × timepoint observations + snapshot eligibility |

In [3]:
firms = load_csv("firms.csv")
inspect(
    "firms.csv",
    firms,
    highlight=["run_id", "firm_id", "company", "event_year", "primary_domain"],
)

FILE: firms.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/firms.csv
shape: 5 rows × 19 columns

Columns:
   1. run_id  ← key
   2. firm_id  ← key
   3. rank
   4. company  ← key
   5. event_type
   6. event_year  ← key
   7. event_label
   8. event_date_inferred
   9. event_date_verified
  10. event_date_final
  11. event_date_precision
  12. event_date_source
  13. event_date_verification_status
  14. source_confidence
  15. wayback_tier
  16. primary_domain  ← key
  17. website
  18. event_source_url
  19. notes

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,rank,company,event_type,event_year,event_label,event_date_inferred,event_date_verified,event_date_final,event_date_precision,event_date_source,event_date_verification_status,source_confidence,wayback_tier,primary_domain,website,event_source_url,notes
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,year,unverified_proxy: event_year=2024 mid-year,unverified_proxy,medium,B,bluemoon.de,https://www.bluemoon.de/,user screening / Orbis export; source URL still to document,Sehr gute Webdeckung; Eventquelle noch sauber extern dokumentieren.
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,2,PETER-LACKE HOLDING GMBH,family_internal_succession,2015,Übernahme durch Sohn in 5. Generation,NaN,NaN,2015-07-01,year,unverified_proxy: event_year=2015 mid-year,unverified_proxy,medium,B,peter-lacke.de,https://www.peter-lacke.de/,user screening / Orbis export; source URL still to document,Sehr gute Webdeckung; familienbezogene Validierung priorisieren.
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,3,3,ANLAGENTECHNIK LEICHTLE GMBH,unclear_family_transition,2023,Übernahme unklar; geringe Text-/Informationsdichte,NaN,NaN,2023-07-01,year,unverified_proxy: event_year=2023 mid-year,unverified_proxy,low,B,anlagentechnik-leichtle.de,https://www.anlagentechnik-leichtle.de/,user screening / Orbis export; source URL still to document,"Nur als Backup, wenn die Eventvalidierung gelingt."
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,4,4,MYRENNE GMBH,family_internal_succession,2012,Übergabe im April 2012 bestätigt,2012-04-15,2012-04-15,2012-04-15,month,event_label: Übergabe im April 2012 bestätigt,verified,high,A,myrenne.com,https://myrenne.com/,user screening / Orbis export; source URL still to document,Wayback funktioniert ohne /de besser; guter Pilotcase.
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,5,5,MSF-VATHAUER ANTRIEBSTECHNIK GMBH & CO. KG,family_internal_succession_or_management_entry,2006,Marc Vathauer tritt in die Geschäftsführung ein,NaN,2006-01-01,2006-01-01,year,event_year; MSF-Vathauer management entry 2006 (exact day unknown),verified,high,A,msf-technik.de,https://www.msf-technik.de/,https://www.msf-technik.de/historie,Nicht als 2010-Event nutzen; für 2006 gute Pre/Post-Waybackdeckung.


In [4]:
snapshots = load_csv("snapshots.csv")
inspect(
    "snapshots.csv",
    snapshots,
    highlight=[
        "run_id",
        "firm_id",
        "company",
        "relative_timepoint",
        "snapshot_status",
        "observation_recommendation",
        "analysis_eligible",
        "selected_capture_date",
    ],
)

FILE: snapshots.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/snapshots.csv
shape: 25 rows × 50 columns

Columns:
   1. run_id  ← key
   2. firm_id  ← key
   3. rank
   4. company  ← key
   5. event_type
   6. event_year
   7. event_label
   8. event_date_inferred
   9. event_date_verified
  10. event_date_final
  11. event_date_precision
  12. event_date_source
  13. event_date_verification_status
  14. relative_timepoint  ← key
  15. target_year
  16. target_date
  17. target_date_precision
  18. observation_is_future
  19. snapshot_status  ← key
  20. capture_source
  21. requested_url
  22. cdx_original_url
  23. canonical_original_url
  24. archive_timestamp
  25. selected_capture_date  ← key
  26. temporal_distance_days
  27. temporal_fit_quality
  28. temporal_fit_usable_default
  29. event_snapshot_position
  30. days_from_actual_event
  31. days_between_selected_snapshots
  32. adjacent_period_overlap_flag
  33. homepage_available
  34. relevant_subpag

,run_id,firm_id,rank,company,event_type,event_year,event_label,event_date_inferred,event_date_verified,event_date_final,event_date_precision,event_date_source,event_date_verification_status,relative_timepoint,target_year,target_date,target_date_precision,observation_is_future,snapshot_status,capture_source,...,days_between_selected_snapshots,adjacent_period_overlap_flag,homepage_available,relevant_subpages_available,subpage_only_observation,observation_scope,observation_recommendation,analysis_eligible,duplicate_capture_flag,duplicate_capture_winner_timepoint,wayback_replay_url,http_status,mime_type,digest,redirect_chain,selection_reason,fallback_attempts,failure_reason,event_year_precision_note,discovered_at
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,year,unverified_proxy: event_year=2024 mid-year,unverified_proxy,pre_pre_event,2020,2020-07-01,year,False,selected,wayback,...,NaN,False,True,False,False,homepage_only,include,True,False,NaN,https://web.archive.org/web/20201101073754id_/https://www.bluemoon.de/,200.0,text/html,UJKZA74AUROG7X3A6VRTKNPEM5RH6ZDL,NaN,closest_within_548d_later_homepage,"[""http://bluemoon.de/"", ""http://www.bluemoon.de/"", ""https://bluemoon.de/"", ""...",NaN,NaN,2026-07-16T09:14:20.634599+00:00
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,year,unverified_proxy: event_year=2024 mid-year,unverified_proxy,pre_event,2022,2022-07-01,year,False,selected,wayback,...,613.0,False,True,False,False,homepage_only,include,True,False,NaN,https://web.archive.org/web/20220707114701id_/https://www.bluemoon.de/,200.0,text/html,7ZPOWU7LMOWNDUXSHN2KTPCYBWOUVO3Q,NaN,closest_within_548d_later_homepage,"[""http://bluemoon.de/"", ""http://www.bluemoon.de/"", ""https://bluemoon.de/"", ""...",NaN,NaN,2026-07-16T09:14:20.658082+00:00
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,year,unverified_proxy: event_year=2024 mid-year,unverified_proxy,event,2024,2024-07-01,year,False,selected,wayback,...,981.0,False,True,False,False,homepage_only,include,True,False,NaN,https://web.archive.org/web/20250314235344id_/https://www.bluemoon.de/,200.0,text/html,HMTV2LAOGVS4YEQ765UZSHMKPZDFPVVE,NaN,closest_within_548d_later_homepage,"[""http://bluemoon.de/"", ""http://www.bluemoon.de/"", ""https://bluemoon.de/"", ""...",NaN,Event target uses year-level precision (2024-07-01). Event date source: unve...,2026-07-16T09:14:20.659718+00:00
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,year,unverified_proxy: event_year=2024 mid-year,unverified_proxy,post_event,2026,2026-07-01,year,False,selected,wayback,...,355.0,False,True,False,False,homepage_only,include,True,False,NaN,https://web.archive.org/web/20260304002641id_/https://www.bluemoon.de/,200.0,text/html,64AJ5PJEZHUCRYIZ2AZV33K6RYD5D77W,NaN,closest_within_548d_earlier_homepage,"[""http://bluemoon.de/"", ""http://www.bluemoon.de/"", ""https://bluemoon.de/"", ""...",NaN,NaN,2026-07-16T09:14:20.661247+00:00
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,year,unverified_proxy: event_year=2024 mid-year,unverified_proxy,post_post_event,2028,2028-07-01,year,True,future_unavailable,wayback,...,NaN,False,False,False,False,unavailable,exclude,False,False,NaN,NaN,NaN,NaN,NaN,NaN,future_unavailable,NaN,NaN,NaN,2026-07-16T09:14:20.661342+00:00


---
## 2. Page-level extraction + classification (v1.1 additions)

`pages.csv` is the full page table.

**New / important columns:** `page_category`, `branding_corpus_eligible`, `governance_metadata_eligible`, `text_language*`, `token_count`, `token_count_reason`

In [5]:
pages = load_csv("pages.csv")
inspect(
    "pages.csv",
    pages,
    highlight=[
        "run_id",
        "firm_id",
        "relative_timepoint",
        "page_category",
        "page_category_reason",
        "branding_corpus_eligible",
        "branding_corpus_exclusion_reason",
        "governance_metadata_eligible",
        "usable_for_analysis",
        "text_language",
        "text_language_confidence",
        "token_count",
        "token_count_reason",
        "word_count",
        "main_text",
    ],
)

FILE: pages.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/pages.csv
shape: 394 rows × 78 columns

Columns:
   1. run_id  ← key
   2. firm_id  ← key
   3. rank
   4. company
   5. event_type
   6. event_year
   7. event_label
   8. relative_timepoint  ← key
   9. target_year
  10. target_date
  11. observation_is_future
  12. snapshot_status
  13. snapshot_observation_key
  14. analysis_eligible
  15. capture_source
  16. archive_timestamp
  17. page_archive_timestamp
  18. selected_capture_date
  19. temporal_distance_days
  20. temporal_fit_quality
  21. wayback_replay_url
  22. original_archived_url
  23. requested_url
  24. final_url
  25. redirect_chain
  26. http_status
  27. mime_type
  28. fetch_timestamp
  29. fetch_error
  30. content_hash
  31. scheme
  32. hostname
  33. subdomain
  34. registrable_domain
  35. suffix
  36. normalized_url
  37. path
  38. query
  39. crawl_depth
  40. discovered_from_url
  41. page_priority_reason
  42. page_category

,run_id,firm_id,rank,company,event_type,event_year,event_label,relative_timepoint,target_year,target_date,observation_is_future,snapshot_status,snapshot_observation_key,analysis_eligible,capture_source,archive_timestamp,page_archive_timestamp,selected_capture_date,temporal_distance_days,temporal_fit_quality,...,main_text,visible_text,extracted_text,extraction_method,text_language,text_language_confidence,text_language_method,text_language_reason,character_count,token_count,token_count_reason,word_count,extraction_quality_score,boilerplate_ratio,duplicate_content_flag,soft_404_flag,archive_toolbar_removed_flag,likely_navigation_only,usable_for_analysis,exclusion_reason
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,+49 2131 66156-0 info[at]bluemoon.de Newsletter Blog News Lexikon Deutsch En...,Full Service Werbeagentur Neuss bei Düsseldorf | BLUE MOON +49 2131 66156-0 ...,+49 2131 66156-0 info[at]bluemoon.de Newsletter Blog News Lexikon Deutsch En...,playwright,de,1.0,lingua,NaN,12502.0,1644.0,NaN,1644.0,0.665,0.000,False,False,False,False,True,NaN
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,+49 2131 66156-0 info[at]bluemoon.de Newsletter Blog News Lexikon Deutsch Pr...,PR News: Werbeagentur | BLUE MOON +49 2131 66156-0 info[at]bluemoon.de Newsl...,+49 2131 66156-0 info[at]bluemoon.de Newsletter Blog News Lexikon Deutsch Pr...,playwright,de,1.0,lingua,NaN,4732.0,618.0,NaN,618.0,0.694,0.000,False,False,False,False,True,NaN
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,TEXTS VIDEO AUDIO SOFTWARE IMAGES SIGN UP | LOG IN UPLOAD Sign up for free L...,toggled by interacting with this icon. Internet Archive logo A line drawing ...,TEXTS VIDEO AUDIO SOFTWARE IMAGES SIGN UP | LOG IN UPLOAD Sign up for free L...,playwright,en,1.0,lingua,NaN,305.0,49.0,NaN,49.0,0.397,0.000,False,False,True,False,True,NaN
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,Search the history of more than 1 trillion web pages. Capture a web page as ...,toggled by interacting with this icon. Internet Archive logo A line drawing ...,Search the history of more than 1 trillion web pages. Capture a web page as ...,trafilatura,en,1.0,lingua,NaN,263.0,46.0,NaN,46.0,0.030,0.934,False,False,True,False,True,NaN
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,Neue Digital-Förderung für den Mittelstand Für eine erfolgversprechende Zuku...,Neue Digital-Förderung für den Mittelstand - BLUE MOON Social Media PR Agent...,Neue Digital-Förderung für den Mittelstand Für eine erfolgversprechende Zuku...,trafilatura,de,1.0,lingua,NaN,3089.0,386.0,NaN,386.0,0.550,0.171,False,False,False,False,True,NaN


In [6]:
# Quick classification / language / token sanity checks
print("page_category counts:")
display(pages["page_category"].value_counts(dropna=False))

print("\nbranding_corpus_eligible:")
display(pages["branding_corpus_eligible"].value_counts(dropna=False))

print("\ntext_language:")
display(pages["text_language"].value_counts(dropna=False))

has_text = pages["main_text"].fillna("").astype(str).str.strip().ne("")
zero_tokens = pages["token_count"].fillna(0).eq(0)
print("\nrows with non-empty main_text but token_count == 0:", int((has_text & zero_tokens).sum()))

page_category counts:


page_category
news_press           97
contact              76
privacy_policy       71
search_archive       35
navigation_only      26
impressum            24
homepage             22
products_services    19
careers_employer     16
family_values         4
technical_system      4
Name: count, dtype: int64


branding_corpus_eligible:


branding_corpus_eligible
True     231
False    163
Name: count, dtype: int64


text_language:


text_language
de         335
en          44
NaN          8
unknown      7
Name: count, dtype: int64


rows with non-empty main_text but token_count == 0: 0


---
## 3. Branding / NLP corpus

Legal/technical pages excluded. Use these for family-branding NLP.

| File | Role |
|------|------|
| `branding_corpus_pages.csv` | Eligible pages only |
| `branding_corpus_observations.csv` | Aggregated text per firm × timepoint |
| `branding_corpus_observations_primary.csv` | Primary analysis pool |
| `branding_corpus_observations_sensitivity.csv` | Sensitivity pool |

In [7]:
branding_pages = load_csv("branding_corpus_pages.csv")
inspect(
    "branding_corpus_pages.csv",
    branding_pages,
    highlight=[
        "firm_id",
        "relative_timepoint",
        "page_category",
        "branding_corpus_eligible",
        "token_count",
        "text_language",
        "main_text",
    ],
)

FILE: branding_corpus_pages.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/branding_corpus_pages.csv
shape: 231 rows × 80 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. rank
   4. company
   5. event_type
   6. event_year
   7. event_label
   8. relative_timepoint  ← key
   9. target_year
  10. target_date
  11. observation_is_future
  12. snapshot_status
  13. snapshot_observation_key
  14. analysis_eligible
  15. capture_source
  16. archive_timestamp
  17. page_archive_timestamp
  18. selected_capture_date
  19. temporal_distance_days
  20. temporal_fit_quality
  21. wayback_replay_url
  22. original_archived_url
  23. requested_url
  24. final_url
  25. redirect_chain
  26. http_status
  27. mime_type
  28. fetch_timestamp
  29. fetch_error
  30. content_hash
  31. scheme
  32. hostname
  33. subdomain
  34. registrable_domain
  35. suffix
  36. normalized_url
  37. path
  38. query
  39. crawl_depth
  40. discovered_from_url
  41. page_priority_r

,run_id,firm_id,rank,company,event_type,event_year,event_label,relative_timepoint,target_year,target_date,observation_is_future,snapshot_status,snapshot_observation_key,analysis_eligible,capture_source,archive_timestamp,page_archive_timestamp,selected_capture_date,temporal_distance_days,temporal_fit_quality,...,extracted_text,extraction_method,text_language,text_language_confidence,text_language_method,text_language_reason,character_count,token_count,token_count_reason,word_count,extraction_quality_score,boilerplate_ratio,duplicate_content_flag,soft_404_flag,archive_toolbar_removed_flag,likely_navigation_only,usable_for_analysis,exclusion_reason,text_analysis_eligible,text_analysis_quality_band
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,+49 2131 66156-0 info[at]bluemoon.de Newsletter Blog News Lexikon Deutsch En...,playwright,de,1.0,lingua,NaN,12502.0,1644.0,NaN,1644.0,0.665,0.000,False,False,False,False,True,NaN,True,high
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,+49 2131 66156-0 info[at]bluemoon.de Newsletter Blog News Lexikon Deutsch Pr...,playwright,de,1.0,lingua,NaN,4732.0,618.0,NaN,618.0,0.694,0.000,False,False,False,False,True,NaN,True,high
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,Neue Digital-Förderung für den Mittelstand Für eine erfolgversprechende Zuku...,trafilatura,de,1.0,lingua,NaN,3089.0,386.0,NaN,386.0,0.550,0.171,False,False,False,False,True,NaN,True,high
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,Immer wieder verfolgt uns in den letzten Jahren die Vorhersage des endgültge...,trafilatura,de,1.0,lingua,NaN,2499.0,313.0,NaN,313.0,0.505,0.306,False,False,False,False,True,NaN,True,high
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,pre_pre_event,2020,2020-07-01,False,selected,1|pre_pre_event,True,wayback,2.020110e+13,2.020110e+13,2020-11-01,123.0,moderate,...,Impressum BLUE MOON Communication Consultants GmbH Friedrichstraße 8 D-41460...,readability,de,1.0,lingua,NaN,1740.0,215.0,NaN,215.0,0.473,0.436,False,False,False,False,True,NaN,True,high


In [8]:
# Confirm no legal/technical / impressum categories leaked into branding corpus
banned = {
    "privacy_policy",
    "terms_conditions",
    "cookie_notice",
    "legal_other",
    "technical_system",
    "search_archive",
    "navigation_only",
    "impressum",
}
leaks = branding_pages[branding_pages["page_category"].isin(banned)]
print("Banned categories in branding_corpus_pages:", len(leaks))
if len(leaks):
    display(leaks[["firm_id", "relative_timepoint", "page_category", "original_archived_url"]].head())
else:
    print("✓ No banned categories found")

print("\nCategories present in branding corpus:")
display(branding_pages["page_category"].value_counts())

Banned categories in branding_corpus_pages: 0
✓ No banned categories found

Categories present in branding corpus:


page_category
news_press           97
contact              76
homepage             19
products_services    19
careers_employer     16
family_values         4
Name: count, dtype: int64

In [9]:
branding_obs = load_csv("branding_corpus_observations.csv")
inspect(
    "branding_corpus_observations.csv",
    branding_obs,
    highlight=[
        "firm_id",
        "company",
        "relative_timepoint",
        "branding_word_count",
        "branding_token_count",
        "source_page_count",
        "text_analysis_eligible",
        "text_analysis_quality_band",
        "branding_text",
    ],
)

FILE: branding_corpus_observations.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/branding_corpus_observations.csv
shape: 25 rows × 12 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. company  ← key
   4. relative_timepoint  ← key
   5. branding_text  ← key
   6. branding_word_count  ← key
   7. branding_token_count  ← key
   8. source_page_count  ← key
   9. source_page_urls_json
  10. source_page_categories_json
  11. text_analysis_eligible  ← key
  12. text_analysis_quality_band  ← key

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,company,relative_timepoint,branding_text,branding_word_count,branding_token_count,source_page_count,source_page_urls_json,source_page_categories_json,text_analysis_eligible,text_analysis_quality_band
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,BLUE MOON: DEINE FULL SERVICE WERBEAGENTUR. Familiär. Zuverlässig. State of ...,7870,7870,11,"[""https://www.bluemoon.de/agentur"", ""https://www.bluemoon.de/"", ""https://www...","[""family_values"", ""homepage"", ""news_press"", ""news_press"", ""news_press"", ""new...",True,high
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,"WE CREATE CONTENT THAT BOOSTS Social Media ist längst kein Hype mehr, sonder...",8021,8021,13,"[""https://www.bluemoon.de/social-media"", ""https://www.bluemoon.de/agentur"", ...","[""contact"", ""family_values"", ""homepage"", ""news_press"", ""news_press"", ""news_p...",True,high
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,event,BLUE MOON at work – unsere Cases Erfolgreich für Champions Wir laden dich ei...,9658,9658,23,"[""https://www.bluemoon.de/cases"", ""https://www.bluemoon.de/jobs/pr-consultan...","[""careers_employer"", ""careers_employer"", ""careers_employer"", ""careers_employ...",True,high
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_event,Social Media PR Agentur Online Marketing Branding Agentur Cases Karriere Blo...,12659,12659,23,"[""https://www.bluemoon.de/cases"", ""https://www.bluemoon.de/jobs/pr-consultan...","[""careers_employer"", ""careers_employer"", ""careers_employer"", ""careers_employ...",True,high
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_post_event,NaN,0,0,0,NaN,NaN,False,ineligible


In [10]:
branding_primary = load_csv("branding_corpus_observations_primary.csv")
inspect(
    "branding_corpus_observations_primary.csv",
    branding_primary,
    highlight=[
        "firm_id",
        "relative_timepoint",
        "branding_token_count",
        "text_analysis_eligible",
        "text_analysis_quality_band",
    ],
)

FILE: branding_corpus_observations_primary.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/branding_corpus_observations_primary.csv
shape: 16 rows × 12 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. company
   4. relative_timepoint  ← key
   5. branding_text
   6. branding_word_count
   7. branding_token_count  ← key
   8. source_page_count
   9. source_page_urls_json
  10. source_page_categories_json
  11. text_analysis_eligible  ← key
  12. text_analysis_quality_band  ← key

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,company,relative_timepoint,branding_text,branding_word_count,branding_token_count,source_page_count,source_page_urls_json,source_page_categories_json,text_analysis_eligible,text_analysis_quality_band
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,BLUE MOON: DEINE FULL SERVICE WERBEAGENTUR. Familiär. Zuverlässig. State of ...,7870,7870,11,"[""https://www.bluemoon.de/agentur"", ""https://www.bluemoon.de/"", ""https://www...","[""family_values"", ""homepage"", ""news_press"", ""news_press"", ""news_press"", ""new...",True,high
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,"WE CREATE CONTENT THAT BOOSTS Social Media ist längst kein Hype mehr, sonder...",8021,8021,13,"[""https://www.bluemoon.de/social-media"", ""https://www.bluemoon.de/agentur"", ...","[""contact"", ""family_values"", ""homepage"", ""news_press"", ""news_press"", ""news_p...",True,high
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,event,BLUE MOON at work – unsere Cases Erfolgreich für Champions Wir laden dich ei...,9658,9658,23,"[""https://www.bluemoon.de/cases"", ""https://www.bluemoon.de/jobs/pr-consultan...","[""careers_employer"", ""careers_employer"", ""careers_employer"", ""careers_employ...",True,high
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_event,Social Media PR Agentur Online Marketing Branding Agentur Cases Karriere Blo...,12659,12659,23,"[""https://www.bluemoon.de/cases"", ""https://www.bluemoon.de/jobs/pr-consultan...","[""careers_employer"", ""careers_employer"", ""careers_employer"", ""careers_employ...",True,high
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,PETER-LACKE HOLDING GMBH,pre_pre_event,"PETER-LACKE Farbe & mehr Direkt zum Hauptmenü, zum Inhalt. Servicemenü Selec...",2631,2631,12,"[""http://www.peter-lacke.de/deu/peter-lacke/unternehmen/karriere/karriere.ht...","[""careers_employer"", ""contact"", ""contact"", ""contact"", ""homepage"", ""news_pres...",True,high


In [11]:
branding_sensitivity = load_csv("branding_corpus_observations_sensitivity.csv")
inspect(
    "branding_corpus_observations_sensitivity.csv",
    branding_sensitivity,
    highlight=[
        "firm_id",
        "relative_timepoint",
        "branding_token_count",
        "text_analysis_eligible",
        "text_analysis_quality_band",
    ],
)

FILE: branding_corpus_observations_sensitivity.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/branding_corpus_observations_sensitivity.csv
shape: 19 rows × 12 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. company
   4. relative_timepoint  ← key
   5. branding_text
   6. branding_word_count
   7. branding_token_count  ← key
   8. source_page_count
   9. source_page_urls_json
  10. source_page_categories_json
  11. text_analysis_eligible  ← key
  12. text_analysis_quality_band  ← key

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,company,relative_timepoint,branding_text,branding_word_count,branding_token_count,source_page_count,source_page_urls_json,source_page_categories_json,text_analysis_eligible,text_analysis_quality_band
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,BLUE MOON: DEINE FULL SERVICE WERBEAGENTUR. Familiär. Zuverlässig. State of ...,7870,7870,11,"[""https://www.bluemoon.de/agentur"", ""https://www.bluemoon.de/"", ""https://www...","[""family_values"", ""homepage"", ""news_press"", ""news_press"", ""news_press"", ""new...",True,high
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,"WE CREATE CONTENT THAT BOOSTS Social Media ist längst kein Hype mehr, sonder...",8021,8021,13,"[""https://www.bluemoon.de/social-media"", ""https://www.bluemoon.de/agentur"", ...","[""contact"", ""family_values"", ""homepage"", ""news_press"", ""news_press"", ""news_p...",True,high
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,event,BLUE MOON at work – unsere Cases Erfolgreich für Champions Wir laden dich ei...,9658,9658,23,"[""https://www.bluemoon.de/cases"", ""https://www.bluemoon.de/jobs/pr-consultan...","[""careers_employer"", ""careers_employer"", ""careers_employer"", ""careers_employ...",True,high
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_event,Social Media PR Agentur Online Marketing Branding Agentur Cases Karriere Blo...,12659,12659,23,"[""https://www.bluemoon.de/cases"", ""https://www.bluemoon.de/jobs/pr-consultan...","[""careers_employer"", ""careers_employer"", ""careers_employer"", ""careers_employ...",True,high
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,PETER-LACKE HOLDING GMBH,pre_pre_event,"PETER-LACKE Farbe & mehr Direkt zum Hauptmenü, zum Inhalt. Servicemenü Selec...",2631,2631,12,"[""http://www.peter-lacke.de/deu/peter-lacke/unternehmen/karriere/karriere.ht...","[""careers_employer"", ""contact"", ""contact"", ""contact"", ""homepage"", ""news_pres...",True,high


---
## 4. Impressum / governance metadata layer

Separate from the branding corpus. Useful for ownership / management validation.

| File | Role |
|------|------|
| `governance_metadata_pages.csv` | Impressum (and related) pages + raw/structured fields |
| `governance_metadata_observations.csv` | One row per firm × timepoint |

In [12]:
gov_pages = load_csv("governance_metadata_pages.csv")
inspect(
    "governance_metadata_pages.csv",
    gov_pages,
    highlight=[
        "firm_id",
        "relative_timepoint",
        "document_title",
        "extracted_text",
        "managing_directors_raw",
        "legal_representatives_raw",
        "legal_entity_raw",
        "parent_company_raw",
        "extraction_confidence",
    ],
)

FILE: governance_metadata_pages.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/governance_metadata_pages.csv
shape: 24 rows × 19 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. company
   4. relative_timepoint  ← key
   5. target_date
   6. selected_capture_date
   7. original_archived_url
   8. wayback_replay_url
   9. document_title  ← key
  10. extracted_text  ← key
  11. managing_directors_raw  ← key
  12. legal_representatives_raw  ← key
  13. legal_entity_raw  ← key
  14. parent_company_raw  ← key
  15. registered_address_raw
  16. registration_number_raw
  17. vat_id_raw
  18. extraction_confidence  ← key
  19. extraction_notes

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,company,relative_timepoint,target_date,selected_capture_date,original_archived_url,wayback_replay_url,document_title,extracted_text,managing_directors_raw,legal_representatives_raw,legal_entity_raw,parent_company_raw,registered_address_raw,registration_number_raw,vat_id_raw,extraction_confidence,extraction_notes
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,PETER-LACKE HOLDING GMBH,pre_pre_event,2011-07-01,2011-07-09,http://www.peter-lacke.de/deu/peter-lacke/unternehmen/unternehmen.html,https://web.archive.org/web/20110709094353.0id_/http://www.peter-lacke.de/de...,Peter-Lacke - \n\t\tPETER-LACKE Farbe & mehr,Unternehmen - Peter-Lacke Wir sind ein operativ ausgerichtetes Mittelstandsu...,NaN,NaN,Tradition & Innovation Wer mit Ausdauer und Beharrlichkeit innovativ und mar...,NaN,NaN,NaN,NaN,0.25,NaN
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,PETER-LACKE HOLDING GMBH,pre_pre_event,2011-07-01,2011-07-09,http://www.peter-lacke.de/deu/peter-lacke/unternehmen/peter-lacke/peter-lack...,https://web.archive.org/web/20110709094353.0id_/http://www.peter-lacke.de/de...,Peter-Lacke - \n\t\tPETER-LACKE Farbe & mehr,Unternehmen - Peter-Lacke Wir sind ein operativ ausgerichtetes Mittelstandsu...,NaN,NaN,Tradition & Innovation Wer mit Ausdauer und Beharrlichkeit innovativ und mar...,NaN,NaN,NaN,NaN,0.25,NaN
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,PETER-LACKE HOLDING GMBH,pre_pre_event,2011-07-01,2011-07-09,http://www.peter-lacke.de/deu/peter-lacke/unternehmen/unternehmensgruppe/unt...,https://web.archive.org/web/20110709094353.0id_/http://www.peter-lacke.de/de...,Unternehmensgruppe - \n\t\tPETER-LACKE Farbe & mehr,Unternehmen - Unternehmensgruppe Die PETER-LACKE Unternehmensgruppe ist bere...,Andreas Peter von großer Bedeutung,NaN,NaN,NaN,NaN,NaN,NaN,0.25,NaN
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,PETER-LACKE HOLDING GMBH,pre_pre_event,2011-07-01,2011-07-09,http://www.peter-lacke.de/deu/peter-lacke/impressum/impressum.html,https://web.archive.org/web/20110709094353.0id_/http://www.peter-lacke.de/de...,Impressum - \n\t\tPETER-LACKE Farbe & mehr,"PETER-LACKE Farbe & mehr Direkt zum Hauptmenü, zum Inhalt. Servicemenü Selec...",Andreas Peter USt,NaN,national Angaben zum Unternehmen gemäß Telemediengesetz Redaktion & Inhalte ...,NaN,NaN,682 Haftungshinweis Trotz sorgf,NaN,0.75,NaN
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,PETER-LACKE HOLDING GMBH,pre_pre_event,2011-07-01,2011-07-09,http://www.peter-lacke.de/deu,https://web.archive.org/web/20110709094353.0id_/http://www.peter-lacke.de/deu,PETER-LACKE Farbe & mehr,"PETER-LACKE Farbe & mehr Direkt zum Hauptmenü, zum Inhalt. Servicemenü Selec...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,raw_text_retained_only


In [13]:
gov_obs = load_csv("governance_metadata_observations.csv")
inspect(
    "governance_metadata_observations.csv",
    gov_obs,
    highlight=[
        "firm_id",
        "relative_timepoint",
        "impressum_available",
        "governance_metadata_text",
        "detected_legal_representatives",
        "detected_legal_entity",
        "detected_parent_company",
        "governance_metadata_quality",
    ],
)

FILE: governance_metadata_observations.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/governance_metadata_observations.csv
shape: 25 rows × 13 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. company
   4. relative_timepoint  ← key
   5. target_date
   6. selected_capture_date
   7. impressum_available  ← key
   8. governance_metadata_text  ← key
   9. detected_legal_representatives  ← key
  10. detected_legal_entity  ← key
  11. detected_parent_company  ← key
  12. governance_metadata_quality  ← key
  13. governance_metadata_source_urls

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,company,relative_timepoint,target_date,selected_capture_date,impressum_available,governance_metadata_text,detected_legal_representatives,detected_legal_entity,detected_parent_company,governance_metadata_quality,governance_metadata_source_urls
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,2020-07-01,2020-11-01,False,NaN,NaN,NaN,NaN,low,NaN
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,2022-07-01,2022-07-07,False,NaN,NaN,NaN,NaN,low,NaN
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,event,2024-07-01,2025-03-14,False,NaN,NaN,NaN,NaN,low,NaN
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_event,2026-07-01,2026-03-04,False,NaN,NaN,NaN,NaN,low,NaN
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_post_event,2028-07-01,NaN,False,NaN,NaN,NaN,NaN,low,NaN


---
## 5. Observation-level text eligibility

**Key file for NLP inclusion decisions:** `observation_text_summary.csv`

Distinguishes snapshot/analysis eligibility from actual text-volume eligibility.

In [14]:
obs_summary = load_csv("observation_text_summary.csv")
inspect(
    "observation_text_summary.csv",
    obs_summary,
    highlight=[
        "firm_id",
        "company",
        "relative_timepoint",
        "analysis_eligible",
        "n_pages_extraction_usable",
        "n_branding_pages_eligible",
        "branding_word_count",
        "branding_token_count",
        "detected_primary_language",
        "text_analysis_eligible",
        "text_analysis_exclusion_reason",
        "text_analysis_quality_band",
    ],
)

FILE: observation_text_summary.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/observation_text_summary.csv
shape: 25 rows × 28 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. company  ← key
   4. relative_timepoint  ← key
   5. target_date
   6. selected_capture_date
   7. snapshot_status
   8. observation_recommendation
   9. analysis_eligible  ← key
  10. n_pages_total
  11. n_pages_fetch_success
  12. n_pages_extraction_usable  ← key
  13. n_branding_pages_eligible  ← key
  14. n_branding_pages_excluded
  15. n_impressum_pages
  16. n_legal_technical_pages
  17. n_duplicate_pages
  18. total_word_count_all_usable_pages
  19. total_token_count_all_usable_pages
  20. branding_word_count  ← key
  21. branding_token_count  ← key
  22. governance_metadata_word_count
  23. detected_primary_language  ← key
  24. language_distribution_json
  25. n_pages_language_unknown
  26. text_analysis_eligible  ← key
  27. text_analysis_exclusion_reason  ← key
  28. te

,run_id,firm_id,company,relative_timepoint,target_date,selected_capture_date,snapshot_status,observation_recommendation,analysis_eligible,n_pages_total,n_pages_fetch_success,n_pages_extraction_usable,n_branding_pages_eligible,n_branding_pages_excluded,n_impressum_pages,n_legal_technical_pages,n_duplicate_pages,total_word_count_all_usable_pages,total_token_count_all_usable_pages,branding_word_count,branding_token_count,governance_metadata_word_count,detected_primary_language,language_distribution_json,n_pages_language_unknown,text_analysis_eligible,text_analysis_exclusion_reason,text_analysis_quality_band
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,2020-07-01,2020-11-01,selected,include,True,25,25,24,11,13,0,13,0,12121,12121,7870,7870,0,de,"{""de"": 7870}",0,True,NaN,high
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,2022-07-01,2022-07-07,selected,include,True,25,15,15,13,2,0,11,0,11760,11760,8021,8021,0,de,"{""de"": 8021}",0,True,NaN,high
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,event,2024-07-01,2025-03-14,selected,include,True,25,25,25,23,2,0,2,0,13191,13191,9658,9658,0,de,"{""de"": 9658}",0,True,NaN,high
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_event,2026-07-01,2026-03-04,selected,include,True,25,25,25,23,2,0,2,0,16610,16610,12659,12659,0,de,"{""de"": 12659}",0,True,NaN,high
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_post_event,2028-07-01,NaN,future_unavailable,exclude,False,1,1,0,0,0,0,0,0,0,0,0,0,0,unknown,NaN,0,False,no_valid_snapshot,ineligible


In [15]:
print("text_analysis_eligible counts:")
display(obs_summary["text_analysis_eligible"].value_counts(dropna=False))

print("\ntext_analysis_quality_band:")
display(obs_summary["text_analysis_quality_band"].value_counts(dropna=False))

print("\nexclusion reasons:")
display(obs_summary["text_analysis_exclusion_reason"].value_counts(dropna=False))

print("\nCompact firm × timepoint view:")
cols = [
    "company",
    "relative_timepoint",
    "n_branding_pages_eligible",
    "branding_token_count",
    "detected_primary_language",
    "text_analysis_eligible",
    "text_analysis_quality_band",
    "text_analysis_exclusion_reason",
]
display(obs_summary[cols].sort_values(["company", "relative_timepoint"]))

text_analysis_eligible counts:


text_analysis_eligible
True     15
False    10
Name: count, dtype: int64


text_analysis_quality_band:


text_analysis_quality_band
high          15
ineligible    10
Name: count, dtype: int64


exclusion reasons:


text_analysis_exclusion_reason
NaN                             15
no_valid_snapshot                5
insufficient_branding_tokens     2
no_branding_pages                2
duplicate_capture                1
Name: count, dtype: int64


Compact firm × timepoint view:


,company,relative_timepoint,n_branding_pages_eligible,branding_token_count,detected_primary_language,text_analysis_eligible,text_analysis_quality_band,text_analysis_exclusion_reason
12,ANLAGENTECHNIK LEICHTLE GMBH,event,2,3297,de,True,high,NaN
13,ANLAGENTECHNIK LEICHTLE GMBH,post_event,2,3392,de,True,high,NaN
14,ANLAGENTECHNIK LEICHTLE GMBH,post_post_event,0,0,unknown,False,ineligible,no_valid_snapshot
11,ANLAGENTECHNIK LEICHTLE GMBH,pre_event,2,3319,de,True,high,NaN
10,ANLAGENTECHNIK LEICHTLE GMBH,pre_pre_event,1,63,de,False,ineligible,insufficient_branding_tokens
2,BLUE MOON COMMUNICATION CONSULTANTS GMBH,event,23,9658,de,True,high,NaN
3,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_event,23,12659,de,True,high,NaN
4,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_post_event,0,0,unknown,False,ineligible,no_valid_snapshot
1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,13,8021,de,True,high,NaN
0,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,11,7870,de,True,high,NaN


---
## 6. Quality summary

Same file name as v1.0, clarified metrics:

| v1.0 | v1.1 |
|------|------|
| `pages_success` | `pages_fetch_success` |
| `pages_usable` | `pages_extraction_usable` |

Also added: `pages_fetch_failed`, `pages_branding_eligible`, `pages_governance_metadata`, `pages_excluded_legal_technical`, `pages_duplicate`.

In [16]:
quality = load_csv("quality_summary.csv")
inspect(
    "quality_summary.csv",
    quality,
    highlight=[
        "firm_id",
        "relative_timepoint",
        "pages_attempted",
        "pages_fetch_success",
        "pages_fetch_failed",
        "pages_extraction_usable",
        "pages_branding_eligible",
        "pages_governance_metadata",
        "pages_excluded_legal_technical",
        "pages_duplicate",
    ],
)

FILE: quality_summary.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/quality_summary.csv
shape: 25 rows × 18 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. rank
   4. company
   5. relative_timepoint  ← key
   6. target_year
   7. snapshot_status
   8. pages_attempted  ← key
   9. pages_fetch_success  ← key
  10. pages_fetch_failed  ← key
  11. pages_extraction_usable  ← key
  12. pages_branding_eligible  ← key
  13. pages_governance_metadata  ← key
  14. pages_excluded_legal_technical  ← key
  15. pages_duplicate  ← key
  16. pages_excluded
  17. avg_text_chars
  18. flags

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,rank,company,relative_timepoint,target_year,snapshot_status,pages_attempted,pages_fetch_success,pages_fetch_failed,pages_extraction_usable,pages_branding_eligible,pages_governance_metadata,pages_excluded_legal_technical,pages_duplicate,pages_excluded,avg_text_chars,flags
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,2020,selected,25,25,0,24,11,0,13,0,1,3654.2,"[""likely_navigation_only"", ""soft_404_flag""]"
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,2022,selected,25,15,10,15,13,0,11,0,10,3619.0,"[""likely_navigation_only"", ""soft_404_flag""]"
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,event,2024,selected,25,25,0,25,23,0,2,0,0,3928.2,NaN
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_event,2026,selected,25,25,0,25,23,0,2,0,0,4918.2,NaN
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_post_event,2028,future_unavailable,1,1,0,0,0,0,0,0,1,NaN,"[""likely_navigation_only""]"


In [17]:
# Consistency checks
q = quality.copy()
bad_attempted = q["pages_attempted"] != (q["pages_fetch_success"] + q["pages_fetch_failed"])
bad_fetch_vs_usable = q["pages_fetch_success"] < q["pages_extraction_usable"]
bad_usable_vs_branding = q["pages_extraction_usable"] < q["pages_branding_eligible"]

print("attempted != fetch_success + fetch_failed:", int(bad_attempted.sum()))
print("fetch_success < extraction_usable:", int(bad_fetch_vs_usable.sum()))
print("extraction_usable < branding_eligible:", int(bad_usable_vs_branding.sum()))

if bad_attempted.any() or bad_fetch_vs_usable.any() or bad_usable_vs_branding.any():
    display(q[bad_attempted | bad_fetch_vs_usable | bad_usable_vs_branding])
else:
    print("✓ quality_summary consistency checks passed")

attempted != fetch_success + fetch_failed: 0
fetch_success < extraction_usable: 0
extraction_usable < branding_eligible: 0
✓ quality_summary consistency checks passed


---
## 7. Analysis pools & validation samples

Snapshot-level analysis pools (from v1.0 architecture) plus validation tables.

| File | Role |
|------|------|
| `analysis_observations.csv` | Primary snapshot pool |
| `analysis_observations_sensitivity.csv` | Sensitivity snapshot pool |
| `firm_coverage_matrix.csv` | Firm × timepoint recommendation matrix |
| `manual_validation_sample.csv` | Archive-validity judgments |
| `manual_corpus_validation.csv` | New v1.1 corpus/classification sample |

In [18]:
analysis = load_csv("analysis_observations.csv")
inspect(
    "analysis_observations.csv",
    analysis,
    highlight=["firm_id", "relative_timepoint", "observation_recommendation", "analysis_eligible"],
)

FILE: analysis_observations.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/analysis_observations.csv
shape: 16 rows × 27 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. rank
   4. company
   5. event_type
   6. event_year
   7. event_label
   8. event_date_inferred
   9. event_date_verified
  10. event_date_final
  11. event_date_verification_status
  12. relative_timepoint  ← key
  13. target_year
  14. target_date
  15. selected_capture_date
  16. archive_timestamp
  17. temporal_distance_days
  18. temporal_fit_quality
  19. temporal_fit_usable_default
  20. event_snapshot_position
  21. observation_scope
  22. observation_recommendation  ← key
  23. analysis_eligible  ← key
  24. duplicate_capture_flag
  25. canonical_original_url
  26. wayback_replay_url
  27. snapshot_status

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,rank,company,event_type,event_year,event_label,event_date_inferred,event_date_verified,event_date_final,event_date_verification_status,relative_timepoint,target_year,target_date,selected_capture_date,archive_timestamp,temporal_distance_days,temporal_fit_quality,temporal_fit_usable_default,event_snapshot_position,observation_scope,observation_recommendation,analysis_eligible,duplicate_capture_flag,canonical_original_url,wayback_replay_url,snapshot_status
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,unverified_proxy,pre_pre_event,2020,2020-07-01,2020-11-01,2.020110e+13,123.0,moderate,True,before_event,homepage_only,include,True,False,https://www.bluemoon.de/,https://web.archive.org/web/20201101073754id_/https://www.bluemoon.de/,selected
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,unverified_proxy,pre_event,2022,2022-07-01,2022-07-07,2.022071e+13,6.0,high,True,before_event,homepage_only,include,True,False,https://www.bluemoon.de/,https://web.archive.org/web/20220707114701id_/https://www.bluemoon.de/,selected
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,unverified_proxy,event,2024,2024-07-01,2025-03-14,2.025031e+13,256.0,low,True,after_event,homepage_only,include,True,False,https://www.bluemoon.de/,https://web.archive.org/web/20250314235344id_/https://www.bluemoon.de/,selected
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,unverified_proxy,post_event,2026,2026-07-01,2026-03-04,2.026030e+13,119.0,moderate,True,after_event,homepage_only,include,True,False,https://www.bluemoon.de/,https://web.archive.org/web/20260304002641id_/https://www.bluemoon.de/,selected
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,2,PETER-LACKE HOLDING GMBH,family_internal_succession,2015,Übernahme durch Sohn in 5. Generation,NaN,NaN,2015-07-01,unverified_proxy,pre_pre_event,2011,2011-07-01,2011-07-09,2.011071e+13,8.0,high,True,before_event,homepage_only,include,True,False,http://www.peter-lacke.de/,https://web.archive.org/web/20110709094353id_/http://www.peter-lacke.de:80/,selected


In [19]:
analysis_sens = load_csv("analysis_observations_sensitivity.csv")
inspect(
    "analysis_observations_sensitivity.csv",
    analysis_sens,
    highlight=["firm_id", "relative_timepoint", "observation_recommendation", "analysis_eligible"],
)

FILE: analysis_observations_sensitivity.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/analysis_observations_sensitivity.csv
shape: 19 rows × 27 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. rank
   4. company
   5. event_type
   6. event_year
   7. event_label
   8. event_date_inferred
   9. event_date_verified
  10. event_date_final
  11. event_date_verification_status
  12. relative_timepoint  ← key
  13. target_year
  14. target_date
  15. selected_capture_date
  16. archive_timestamp
  17. temporal_distance_days
  18. temporal_fit_quality
  19. temporal_fit_usable_default
  20. event_snapshot_position
  21. observation_scope
  22. observation_recommendation  ← key
  23. analysis_eligible  ← key
  24. duplicate_capture_flag
  25. canonical_original_url
  26. wayback_replay_url
  27. snapshot_status

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,rank,company,event_type,event_year,event_label,event_date_inferred,event_date_verified,event_date_final,event_date_verification_status,relative_timepoint,target_year,target_date,selected_capture_date,archive_timestamp,temporal_distance_days,temporal_fit_quality,temporal_fit_usable_default,event_snapshot_position,observation_scope,observation_recommendation,analysis_eligible,duplicate_capture_flag,canonical_original_url,wayback_replay_url,snapshot_status
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,unverified_proxy,pre_pre_event,2020,2020-07-01,2020-11-01,2.020110e+13,123.0,moderate,True,before_event,homepage_only,include,True,False,https://www.bluemoon.de/,https://web.archive.org/web/20201101073754id_/https://www.bluemoon.de/,selected
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,unverified_proxy,pre_event,2022,2022-07-01,2022-07-07,2.022071e+13,6.0,high,True,before_event,homepage_only,include,True,False,https://www.bluemoon.de/,https://web.archive.org/web/20220707114701id_/https://www.bluemoon.de/,selected
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,unverified_proxy,event,2024,2024-07-01,2025-03-14,2.025031e+13,256.0,low,True,after_event,homepage_only,include,True,False,https://www.bluemoon.de/,https://web.archive.org/web/20250314235344id_/https://www.bluemoon.de/,selected
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,family_internal_succession,2024,Übernahme durch Sohn nach Tod der Mutter,NaN,NaN,2024-07-01,unverified_proxy,post_event,2026,2026-07-01,2026-03-04,2.026030e+13,119.0,moderate,True,after_event,homepage_only,include,True,False,https://www.bluemoon.de/,https://web.archive.org/web/20260304002641id_/https://www.bluemoon.de/,selected
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,2,PETER-LACKE HOLDING GMBH,family_internal_succession,2015,Übernahme durch Sohn in 5. Generation,NaN,NaN,2015-07-01,unverified_proxy,pre_pre_event,2011,2011-07-01,2011-07-09,2.011071e+13,8.0,high,True,before_event,homepage_only,include,True,False,http://www.peter-lacke.de/,https://web.archive.org/web/20110709094353id_/http://www.peter-lacke.de:80/,selected


In [20]:
coverage = load_csv("firm_coverage_matrix.csv")
inspect("firm_coverage_matrix.csv", coverage)

FILE: firm_coverage_matrix.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/firm_coverage_matrix.csv
shape: 5 rows × 8 columns

Columns:
   1. firm_id
   2. rank
   3. company
   4. pre_pre_event
   5. pre_event
   6. event
   7. post_event
   8. post_post_event

First 5 rows:


,firm_id,rank,company,pre_pre_event,pre_event,event,post_event,post_post_event
0,1,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,include,include,include,include,exclude
1,2,2,PETER-LACKE HOLDING GMBH,include,include,sensitivity_analysis,exclude,exclude
2,3,3,ANLAGENTECHNIK LEICHTLE GMBH,include,include,sensitivity_analysis,include,exclude
3,4,4,MYRENNE GMBH,include,include,exclude,include,include
4,5,5,MSF-VATHAUER ANTRIEBSTECHNIK GMBH & CO. KG,include,exclude_duplicate_capture,sensitivity_analysis,include,include


In [21]:
manual = load_csv("manual_validation_sample.csv")
inspect(
    "manual_validation_sample.csv",
    manual,
    highlight=[
        "firm_id",
        "relative_timepoint",
        "correct_company",
        "valid_archived_page",
        "temporally_appropriate",
        "content_extraction_usable",
        "duplicate_capture",
    ],
)

FILE: manual_validation_sample.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/manual_validation_sample.csv
shape: 19 rows × 19 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. company
   4. relative_timepoint  ← key
   5. target_date
   6. selected_capture_date
   7. temporal_fit_quality
   8. observation_scope
   9. observation_recommendation
  10. original_archived_url
  11. wayback_replay_url
  12. duplicate_capture_flag
  13. correct_company  ← key
  14. valid_archived_page  ← key
  15. temporally_appropriate  ← key
  16. content_extraction_usable  ← key
  17. duplicate_capture  ← key
  18. reviewer_notes
  19. validation_screenshot_path

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,company,relative_timepoint,target_date,selected_capture_date,temporal_fit_quality,observation_scope,observation_recommendation,original_archived_url,wayback_replay_url,duplicate_capture_flag,correct_company,valid_archived_page,temporally_appropriate,content_extraction_usable,duplicate_capture,reviewer_notes,validation_screenshot_path
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,2020-07-01,2020-11-01,moderate,homepage_only,include,https://www.bluemoon.de/,https://web.archive.org/web/20201101073754id_/https://www.bluemoon.de/,False,True,True,True,NaN,False,Pipeline: 1687 words via playwright; title='Full Service Werbeagentur Neuss ...,validation_screenshots/01_1_blue_moon_communication__pre_pre_event.png
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,2022-07-01,2022-07-07,high,homepage_only,include,https://www.bluemoon.de/,https://web.archive.org/web/20220707114701id_/https://www.bluemoon.de/,False,True,True,True,NaN,False,Pipeline: 1641 words via playwright; title='Full Service Werbeagentur Neuss ...,validation_screenshots/02_1_blue_moon_communication__pre_event.png
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,event,2024-07-01,2025-03-14,low,homepage_only,include,https://www.bluemoon.de/,https://web.archive.org/web/20250314235344id_/https://www.bluemoon.de/,False,True,True,False,NaN,False,Pipeline: 501 words via trafilatura; title='Die Werbeagentur des Mittelstand...,validation_screenshots/03_1_blue_moon_communication__event.png
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,post_event,2026-07-01,2026-03-04,moderate,homepage_only,include,https://www.bluemoon.de/,https://web.archive.org/web/20260304002641id_/https://www.bluemoon.de/,False,True,True,True,NaN,False,Pipeline: 1146 words via playwright; title='Die Werbeagentur des Mittelstand...,validation_screenshots/04_1_blue_moon_communication__post_event.png
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,2,PETER-LACKE HOLDING GMBH,pre_pre_event,2011-07-01,2011-07-09,high,homepage_only,include,http://www.peter-lacke.de/,https://web.archive.org/web/20110709094353id_/http://www.peter-lacke.de:80/,False,True,True,True,NaN,False,Pipeline: 241 words via playwright; title='PETER-LACKE Farbe & mehr'. Brow...,validation_screenshots/05_2_peter_lacke_holding_gmbh_pre_pre_event.png


In [22]:
manual_corpus = load_csv("manual_corpus_validation.csv")
inspect(
    "manual_corpus_validation.csv",
    manual_corpus,
    highlight=[
        "firm_id",
        "relative_timepoint",
        "page_category",
        "page_category_correct",
        "branding_corpus_eligibility_correct",
        "legal_technical_exclusion_correct",
        "impressum_layer_correct",
        "token_count_plausible",
        "language_detection_plausible",
        "observation_text_eligibility_correct",
    ],
)

FILE: manual_corpus_validation.csv
path: /Users/rick/DoksOhneidrive/FFB/data/releases/pilot_v1_1/data/manual_corpus_validation.csv
shape: 43 rows × 18 columns

Columns:
   1. run_id
   2. firm_id  ← key
   3. company
   4. relative_timepoint  ← key
   5. target_date
   6. selected_capture_date
   7. page_url
   8. wayback_replay_url
   9. page_category  ← key
  10. page_category_correct  ← key
  11. branding_corpus_eligibility_correct  ← key
  12. legal_technical_exclusion_correct  ← key
  13. impressum_layer_correct  ← key
  14. token_count_plausible  ← key
  15. language_detection_plausible  ← key
  16. observation_text_eligibility_correct  ← key
  17. reviewer_notes
  18. evidence_url

✓ All highlighted columns present

First 5 rows:


,run_id,firm_id,company,relative_timepoint,target_date,selected_capture_date,page_url,wayback_replay_url,page_category,page_category_correct,branding_corpus_eligibility_correct,legal_technical_exclusion_correct,impressum_layer_correct,token_count_plausible,language_detection_plausible,observation_text_eligibility_correct,reviewer_notes,evidence_url
0,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,2020-07-01,2020-11-01,https://www.bluemoon.de/,https://web.archive.org/web/20201101073754.0id_/https://www.bluemoon.de/,homepage,True,True,False,True,True,True,True,Sampled homepage page; branding_eligible=True; tokens=1644.0,https://web.archive.org/web/20201101073754.0id_/https://www.bluemoon.de/
1,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,2020-07-01,2020-11-01,https://www.bluemoon.de/agentur/karriere,https://web.archive.org/web/20201101073754.0id_/https://www.bluemoon.de/agen...,privacy_policy,True,False,True,True,True,True,True,Sampled privacy_policy page; branding_eligible=False; tokens=388.0,https://web.archive.org/web/20201101073754.0id_/https://www.bluemoon.de/agen...
2,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_pre_event,2020-07-01,2020-11-01,https://www.bluemoon.de/lexikon,https://web.archive.org/web/20201101073754.0id_/https://www.bluemoon.de/lexikon,news_press,True,True,False,True,True,True,True,Sampled news_press page; branding_eligible=True; tokens=2315.0,https://web.archive.org/web/20201101073754.0id_/https://www.bluemoon.de/lexikon
3,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,2022-07-01,2022-07-07,https://www.bluemoon.de/,https://web.archive.org/web/20220707114701.0id_/https://www.bluemoon.de/,homepage,True,True,False,True,True,True,True,Sampled homepage page; branding_eligible=True; tokens=1404.0,https://web.archive.org/web/20220707114701.0id_/https://www.bluemoon.de/
4,c956bd4e-2b8a-4c48-8780-c187587f46b3,1,BLUE MOON COMMUNICATION CONSULTANTS GMBH,pre_event,2022-07-01,2022-07-07,https://www.bluemoon.de/agentur/karriere,https://web.archive.org/web/20220707114701.0id_/https://www.bluemoon.de/agen...,privacy_policy,True,False,True,True,True,True,True,Sampled privacy_policy page; branding_eligible=False; tokens=388.0,https://web.archive.org/web/20220707114701.0id_/https://www.bluemoon.de/agen...


---
## 8. Optional cross-checks

Light joins to confirm the layers connect on `firm_id` + `relative_timepoint`.

In [23]:
def key(df: pd.DataFrame) -> pd.Series:
    return df["firm_id"].astype(str) + "|" + df["relative_timepoint"].astype(str)


snap_keys = set(key(snapshots))
obs_keys = set(key(obs_summary))
brand_keys = set(key(branding_obs))
gov_keys = set(key(gov_obs))

print("snapshots observations:", len(snap_keys))
print("observation_text_summary observations:", len(obs_keys))
print("branding_corpus_observations observations:", len(brand_keys))
print("governance_metadata_observations observations:", len(gov_keys))

print("\nobs_summary missing from snapshots:", sorted(obs_keys - snap_keys)[:10])
print("branding_obs missing from snapshots:", sorted(brand_keys - snap_keys)[:10])
print("gov_obs missing from snapshots:", sorted(gov_keys - snap_keys)[:10])

# Governance pages should be traceable to pages.csv URLs
page_urls = set(pages["original_archived_url"].dropna().astype(str))
gov_urls = set(gov_pages["original_archived_url"].dropna().astype(str))
print("\ngovernance page URLs not in pages.csv:", len(gov_urls - page_urls))

snapshots observations: 25
observation_text_summary observations: 25
branding_corpus_observations observations: 25
governance_metadata_observations observations: 25

obs_summary missing from snapshots: []
branding_obs missing from snapshots: []
gov_obs missing from snapshots: []

governance page URLs not in pages.csv: 0


---
## 9. Cheat sheet — what to open

| # | Requirement | Primary file(s) |
|---|-----------------|-----------------|
| 1 | Exclude legal/technical; keep Impressum separate | `branding_corpus_*.csv`, `governance_metadata_*.csv` |
| 2 | Observation text eligibility | `observation_text_summary.csv` |
| 3 | Quality summary consistency | `quality_summary.csv` (`pages_fetch_success`, `pages_extraction_usable`) |
| 4 | Tokens + language | `pages.csv` (`token_count`, `text_language*`) |

**Best starting trio for a quick re-review:**
1. `branding_corpus_observations_primary.csv`
2. `observation_text_summary.csv`
3. `governance_metadata_observations.csv`